## 📖 Libro: §4.2–4.4 del Capítulo 4 — Geod\u00e9sicas \u03b1 en la familia Bernoulli

**Enunciado (verbatim del libro):** *"En una familia exponencial, las geod\u00e9sicas $\alpha = +1$ son rectas en par\u00e1metros naturales $\eta$ y las $\alpha = -1$ son rectas en par\u00e1metros de expectaci\u00f3n $\mu$. La $\alpha = 0$ es la conexi\u00f3n de Levi-Civita (m\u00ednima longitud Riemanniana)."*

**Mini-reto:**
1. En la familia Bernoulli, calcular expl\u00edcitamente las tres geod\u00e9sicas $\alpha = -1, 0, 1$ entre $p_0 = 0.2$ y $p_1 = 0.8$.
2. Visualizarlas en el plano $(\eta, p)$ (donde $\eta = \mathrm{logit}(p)$).
3. Verificar que la $\alpha = -1$ (mixta/convex\u00ednfoga m-proyecci\u00f3n) pasa por el midpoint $p = 0.5$ en $t = 0.5$ en $\eta$ (coordenada de med\u00eda).
4. Verificar que la $\alpha = +1$ (e-proyecci\u00f3n/exponencial) pasa por el midpoint EN $p$ pero en $\eta$ NO es lineal.
5. Comparar longitudes: la $\alpha = 0$ (geod\u00e9sica de Levi-Civita / Fisher) es la m\u00e1s corta en m\u00e9trica de Fisher.

**@ Pregunta a tu LLM:** «¿Por qu\u00e9 la elecci\u00f3n de $\alpha$ corresponde a un compromiso pragm\u00e1tico entre preservar momentos ($\alpha = -1$) o preservar par\u00e1metros naturales ($\alpha = +1$)? ¿Cu\u00e1l es la \u201cmenos sorprendente\u201d matem\u00e1ticamente?»

In [ ]:
# =====================================================================
# Celda 1 — imports + seed determinístico
# =====================================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq

from utils import setup_seed

SEED = setup_seed("cap4_alpha_geodesic_visualizer")
rng = np.random.default_rng(SEED)
print(f"Deterministic seed: {SEED}")

In [ ]:
# =====================================================================
# Celda 2 — Geodésicas α en la familia Bernoulli (Bernoulli² plano)
# =====================================================================
# En familia exponencial: geodésica α entre p0 y p1 está parametrizada por
#
#     p_α(t) = σ^{-1}( (1-t)σ(p0) + t φ_α(p0,p1) ),
#
# donde φ_α controla la conexión. Para Bernoulli como exp-fam con η=logit(p),
# las tres conexiones clásicas son:
#
#     α = -1: m-conexión (recta en μ):  p(t) = (1-t) p0 + t p1
#     α = +1: e-conexión (recta en η): η(t) = (1-t) η(p0) + t η(p1)
#                                          ⇒ p(t) = sigmoid(η(t))
#     α =  0: Levi-Civita — interpolación convexa entre α=±1
#            según src/capitulo4.tex §4.4:
#               p_lecivia(t) = ½·σ(η_t) + ½·((1-t)·p_0 + t·p_1).
#            (Nota: la geodésica Riemann Levi-Civita "verdadera" es el
#            SEMICÍRCULO en coord. arccos(2p-1); el libro adopta la
#            convención lineal por simplicidad pedagógica.)

def eta(p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return np.log(p / (1 - p))

def inv_eta(eta_val):
    return 1.0 / (1.0 + np.exp(-eta_val))

def I_bernoulli(p):
    # Métrica de Fisher (analítica) I(p) = 1/(p(1-p)).
    return 1.0 / (p * (1 - p))

def geodesic_alpha_minus_one(p0, p1, t):
    # m-conexión: recta en μ = p.
    return (1 - t) * p0 + t * p1

def geodesic_alpha_plus_one(p0, p1, t):
    # e-conexión: recta en η = logit, mapeada al simplex.
    eta_t = (1 - t) * eta(p0) + t * eta(p1)
    return inv_eta(eta_t)

def geodesic_alpha_zero(p0, p1, t=None):
    # α = 0 (Levi-Civita): convención src/capitulo4.tex §4.4 — interpolación
    # ½·σ(η_t) + ½·((1-t)·p_0 + t·p_1). En 1-D produce un midpoint p=0.5
    # exacto cuando los endpoints son simétricos.
    eta0 = np.log(p0 / (1 - p0))
    eta1 = np.log(p1 / (1 - p1))
    if t is None:
        t = np.linspace(0, 1, 200)
    p_exp = 1.0 / (1.0 + np.exp(-((1 - t) * eta0 + t * eta1)))
    p_mix = (1 - t) * p0 + t * p1
    return 0.5 * (p_exp + p_mix)

p0, p1 = 0.2, 0.8
ts = np.linspace(0, 1, 50)
g_m1 = np.array([geodesic_alpha_minus_one(p0, p1, t) for t in ts])
g_p1 = np.array([geodesic_alpha_plus_one(p0, p1, t) for t in ts])
g_0 = geodesic_alpha_zero(p0, p1)
_g0_at_05 = geodesic_alpha_zero(p0, p1, t=0.5)  # exact t=0.5 (scalar)

print("Punto medio en t=0.5 (p0=0.2, p1=0.8):")
print("  α=-1 (mixta):       p(0.5) = {:.4f}".format(geodesic_alpha_minus_one(p0, p1, 0.5)))
print("  α=+1 (exponencial): p(0.5) = {:.4f}".format(geodesic_alpha_plus_one(p0, p1, 0.5)))
print("  α=0  (Levi-Civita): p(0.5) = {:.4f}".format(_g0_at_05))
print("\nLectura: las 3 α-curvas pasan por el midpoint p=0.5 por SIMETRÍA (p_0=0.2, p_1=0.8).")


In [ ]:
# =====================================================================
# Celda 3 — Visualizaci\u00f3n de las tres geod\u00e9sicas \u03b1 en el plano (\u03b7, p)
# =====================================================================
fig, ax = plt.subplots(figsize=(10, 5))
etas_all = np.array([eta(p) for p in g_m1] + [eta(p) for p in g_p1] + [eta(p) for p in g_0])
eta_min, eta_max = etas_all.min() - 0.3, etas_all.max() + 0.3
p_grid = np.linspace(0.005, 0.995, 200)
ax.plot(eta(p_grid), p_grid, "black", linewidth=0.7,
        alpha=0.4, label=r"simplex $p = \sigma(\eta)$")

ax.plot([eta(p) for p in g_m1], g_m1, color="#0F766E", linewidth=2.4,
        marker="o", markevery=10, label=r"$\alpha = -1$ (m-conexi\u00f3n / mixta): recta en $p$",
        linestyle="-")
ax.plot([eta(p) for p in g_p1], g_p1, color="#ff5fd2", linewidth=2.4,
        marker="s", markevery=10, label=r"$\alpha = +1$ (e-conexi\u00f3n / expon.): recta en $\eta$",
        linestyle="-")
ax.plot([eta(p) for p in g_0], g_0, color="#8a2be2", linewidth=3.0,
        label=r"$\alpha = 0$  (Levi-Civita / Fisher): semic\u00edrculo en $\eta$",
        linestyle="-")

ax.scatter([eta(p0), eta(p1)], [p0, p1], s=120, zorder=5,
           color=["#FF5733", "#3357FF"], edgecolors="black", linewidth=1.5)
ax.annotate("$p_0 = 0.2$", (eta(p0), p0), textcoords="offset points",
            xytext=(10, -10), fontsize=11)
ax.annotate("$p_1 = 0.8$", (eta(p1), p1), textcoords="offset points",
            xytext=(10, -10), fontsize=11)

ax.set_xlabel(r"$\eta = \mathrm{logit}(p)$  (par\u00e1metro natural)", fontsize=12)
ax.set_ylabel(r"$p$  (par\u00e1metro de expectaci\u00f3n)", fontsize=12)
ax.set_title("Tres geod\u00e9sicas $\\alpha$ entre $p_0=0.2$ y $p_1=0.8$ en la familia Bernoulli",
             fontsize=13)
ax.legend(loc="lower center", fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(eta_min, eta_max)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# Celda 4 — Longitud de cada geodésica en la métrica de Fisher
# =====================================================================
# L(α) = ∫_0^1 sqrt(I(p_α(t)) · (dp_α/dt)²) dt
#       ≡ ∫_0^1 |p_α'(t) / sqrt(I(p_α(t)))| dt   (métrica de Fisher en 1D)
#
# Bajo la convención de este cuaderno: ds² = p(1-p) (dp)²   ⇒  ds = sqrt(p(1-p)) |dp|.
# Push-forward a coord. η=logit:  g_ηη = (dp/dη)² · g_pp = [p(1-p)]³.
# Así ds² = p³(1-p)³ dη² — la métrica NO es constante en η.

def fisher_length(g):
    # Longitud en métrica de Fisher usando integración en p.
    # ds^2 = (dp)^2 / I(p) = p(1-p) (dp)^2  ⇒  ds = sqrt(p(1-p)) |dp|
    dg = np.diff(g)
    p_mid = (g[:-1] + g[1:]) / 2
    return float(np.sum(np.sqrt(p_mid * (1 - p_mid)) * np.abs(dg)))

fisher_len_m1 = fisher_length(g_m1)
fisher_len_p1 = fisher_length(g_p1)
fisher_len_0  = fisher_length(g_0)

eta_diff_abs = abs(eta(p1) - eta(p0))
print("Longitudes en métrica de Fisher (entre p0=0.2 y p1=0.8):")
print("  L(α=-1, mixta)      ≈ {:.4f}".format(fisher_len_m1))
print("  L(α=+1, exponencial) ≈ {:.4f}".format(fisher_len_p1))
print("  L(α=0,  Levi-Civita) ≈ {:.4f}".format(fisher_len_0))

print("\n  |Δη| = |η_1 - η_0| = {:.4f}  ← NO es la longitud de arco, solo el ancho en η".format(eta_diff_abs))

print("\nEn 1-D las 3 α-curvas son 3 PARAMETRIZACIONES DISTINTAS del MISMO")
print("único camino p_0 → p_1; por tanto tienen TODAS idéntica longitud de arco:")
print("  L_uniforme = ∫_{p_0}^{p_1} sqrt(p(1-p)) dp ≈ 0.2813")
print("    (numérico: {:.4f} por trapezoidal sobre la curva representativa)".format(fisher_len_0))
print("Ese valor NO es |Δη| = 2.772: la push-forward g_ηη = [p(1-p)]³ ya NO")
print("es plana en η, así que la coordenada η NO es Riemann-geodésica.")
print("\nLectura final: solo en el caso 1-D de Bernoulli las 3 α-curvas son")
print("arclength-equivalentes. En multiparamétrico (Cap. 4 §4.4, ndim ≥ 2) la")
print("única geodésica estrictamente mínima (Levi-Civita) es α = 0.")


## ✅ `@ Verifica con:`

Las verificaciones que se cumplen:

1. **Geodésicas en t=0.5**:
   - $\alpha = -1$: $p(0.5) = 0.5$ (lineal en $p$: promedio aritmético de 0.2 y 0.8).
   - $\alpha = +1$: $p(0.5) = \sigma((1-t)\eta_0 + t\eta_1)\bigg|_{t=0.5} = \sigma(0) = 0.5$ (sigmoid del logit midpoint).
   - $\alpha = 0$: pasa **exactamente** por $p(0.5) = 0.5$ por SIMETRÍA (también válido bajo una interpretación alternativa de semicírculo arccos: (arccos(2·0.2−1)+arccos(2·0.8−1))/2 = (2.2143+0.9273)/2 = π/2, y cos(π/2)=0 ⇒ 2p(0.5)−1=0 ⇒ p(0.5)=0.5).

2. **Visualización en el plano (η, p)**:
   - $\alpha = -1$ es una línea recta en $p$ mapeada al plano (η, p) vía $p \mapsto \eta$, formando una curva sigmoidal.
   - $\alpha = +1$ es una línea recta en $\eta$, mapeada al plano (η, p) vía la sigmoide $p = \sigma(\eta)$ (curva S en $p$).
   - $\alpha = 0$ (interpretación `capitulo4.tex §4.4`) es la interpolación ½·σ(η_t) + ½·((1−t)·p_0 + t·p_1) — una curva intermedia entre las dos anteriores.

3. **Longitudes en métrica de Fisher**:
   - Bajo la métrica de Fisher en 1-D (Bernoulli), las tres α-curvas son 3 parametrizaciones del MISMO camino $p_0 \to p_1$ ⇒ todas tienen idéntica longitud de arco $\approx 0.2813$ (ver `fisher_len_0` arriba), que **NO** es $|\Delta \eta| = 2.772$.
   - En general (multiparamétrico, no-flat), la α=0 es la única geodésica en el sentido Riemanniano estricto (mínima longitud entre pares).

Conexión con el libro:
- §4.2 (definición α-conexión): las tres curvas representadas son las tres conexiones sobre la misma familia.
- §4.3 (dualidad plana): esta familia particular es dualmente plana porque en coord. mixtas la métrica de Fisher es constante.
- §4.4 (Christoffel dual): los símbolos de Christoffel $\Gamma^{(+\alpha)}$ y $\Gamma^{(-\alpha)}$ son los que producen las trayectorias señaladas.

Discusión para el LLM mentor:
- ¿Cómo modificarías el notebook para la familia Normal($\mu, \sigma^2$)? Las geodésicas $\alpha$ son trayectorias distintas en el plano $(\mu, \sigma)$ que NO son equivalentes en longitud.
- ¿Por qué la métrica de Fisher sobre 1-D Bernoulli hace que las 3 α-curvas coincidan en longitud? Porque en 1-D cualquier camino entre dos puntos tiene longitud única (no hay elección de trayectoria alternativa).
- ¿Qué significaría $\alpha = \pm 2$ (conexiones extendidas)? La fórmula general es interpolación lineal entre $m$ ($\alpha=-1$) y $e$ ($\alpha=+1$); conexiones fuera de $[-1, +1]$ son menos habituales pero formalmente válidas en el marco dual.
